In [62]:
import numpy as np
import neal

def qubo_det_8(terrain, heads, wet_cells, debug=False):
    """
    Deterministic-8 implementation using D-Wave Ocean SDK
    
    Takes 3 3x3 integer-valued grids which represent respectively:
    terrain: a 'Digital Elevation Model' (read from csv file with Geopandas)
    heads: the respective water heads for the terrain (as integers)
    wet_cells: binary grid of cells, 1 if they have water, 0 if not.
    Follows a one-hot scheme.

    Returns:
    A 3x3 grid with the resulting channel network
    The index for the next cell towards were to move
    The steepest descent value for terrain + water
    """
    bench = %timeit -n 1 -r 5  -o [x for x in range(10)] 
    print(f"Best time: {bench.best}")
    t_f, h_f, w_f = [x.flatten() for x in (terrain, heads, wet_cells)]

    aquifer_height = h_f[4] # Take central cell in original terrain 
    if (debug == True):
        print("aquifer initial height: ", aquifer_height.astype(int))
    # We analyze the terrain to develop the channel network
    neighbors_idx = [i for i in range(9)] #if i != 4]
    channel_network = (terrain - t_f[4]).astype(int)
    
    abs_diffs = {i: abs(t_f[i] - (h_f[4] + t_f[4])) for i in neighbors_idx} 
    P = max(abs_diffs.values()) * 2
    qubo = {}
    for i in neighbors_idx:
      qubo[(i, i)] = -abs_diffs[i] - P

    for i in neighbors_idx:
        for j in neighbors_idx:
            if i < j:
                qubo[(i, j)] = 2 * P

    sampler = neal.SimulatedAnnealingSampler()
    response = sampler.sample_qubo(qubo, num_reads=10) 
    best = response.first.sample
    
    destination = [j for j, v in best.items() if v == 1][0]
    if(debug == True):
        print("Sampler Properties:", sampler.properties)
        print("Sampler results: ", response)
    
    return channel_network, destination, abs_diffs[destination]

notation  = np.array([["a00","a01","a02"],["a10","a11","a12"],["a20","a21","a22"]])
terrain = np.array([[6, 5, 5], [7, 2, 5], [0, 4, 6]])
wet_cells = np.array([[0,0,0],[0,1,0],[0,0,0]])
heads = np.array([[0,0,0],[0,3,0],[0,0,0]])

print("\n----------------------------------------------------------------------------------------")
print("QUBO for deterministic-8\n(C) 2026, Jaime Anguiano Olarra (jaimeangola.github@gmail.com)")
print("Simple model for the widely used routine in Hydrology")
print("Read paper for more information (distributed under the GPLv3 (2007)")
print("----------------------------------------------------------------------------------------")
channel_network, destination, extremeval = qubo_det_8(terrain, heads, wet_cells, debug=True)
print(f"Notation, for a general matrix A:\n{notation}")
print(f"Terrain:\n{terrain}\nChannel network:\n{channel_network}\nHeads:\n{heads}\n")
print(f"Max difference: {extremeval}")
row, col = np.unravel_index(destination, terrain.shape)
print(f"Next cell: z_{row}{col}")





----------------------------------------------------------------------------------------
QUBO for deterministic-8
(C) 2026, Jaime Anguiano Olarra (jaimeangola.github@gmail.com)
Simple model for the widely used routine in Hydrology
Read paper for more information (distributed under the GPLv3 (2007)
----------------------------------------------------------------------------------------
The slowest run took 4.25 times longer than the fastest. This could mean that an intermediate result is being cached.
800 ns ± 482 ns per loop (mean ± std. dev. of 5 runs, 1 loop each)
Best time: 4.00003045797348e-07
aquifer initial height:  3
Sampler Properties: {'beta_schedule_options': ('linear', 'geometric', 'custom')}
Sampler results:     0  1  2  3  4  5  6  7  8 energy num_oc.
1  0  0  0  0  0  0  1  0  0  -15.0       1
2  0  0  0  0  0  0  1  0  0  -15.0       1
3  0  0  0  0  0  0  1  0  0  -15.0       1
4  0  0  0  0  0  0  1  0  0  -15.0       1
7  0  0  0  0  0  0  1  0  0  -15.0       1
5  0

In [19]:
res = %timeit -n 1 -r 5  -o [x for x in range(10)]
print(f"Best time: {res.best}")
import numpy as np
from qiskit.quantum_info import Operator

def qiskit_matrix_analysis(matrix_data):
    # Convert input to a Qiskit Operator (matrix representation)
    op = Operator(matrix_data)
    A = op.data # Access the underlying numpy array
    
    # Python 0-based indexing: Center is [1, 1]
    central_val = A[1, 1]
    
    # Subtract central value from all elements
    diff_matrix = A - central_val
    
    # Calculate largest difference among neighbors (excluding the center)
    # Masking center: flatten and remove the 5th element (index 4)
    neighbors = np.delete(diff_matrix.flatten(), 4)
    max_diff = np.max(np.abs(neighbors))
    
    return diff_matrix, max_diff

# Example
mat = np.array([[10, 20, 30], [40, 50, 60], [70, 80, 90]])
diffs, largest = qiskit_matrix_analysis(mat)
print(f"Signed Matrix:\n{diffs}\nLargest Difference: {largest}")


1.14 μs ± 454 ns per loop (mean ± std. dev. of 5 runs, 1 loop each)
Best time: 8.00006091594696e-07
Signed Matrix:
[[-40.+0.j -30.+0.j -20.+0.j]
 [-10.+0.j   0.+0.j  10.+0.j]
 [ 20.+0.j  30.+0.j  40.+0.j]]
Largest Difference: 40.0


In [20]:
res = %timeit -n 1 -r 5  -o [x for x in range(10)]
print(f"Best time: {res.best}")
import numpy as np
import dimod

def solve_max_diff_exact(matrix):
    # 1. Classical setup (Python 0-based indexing)
    flat = matrix.flatten()
    center_val = flat[4]  # Central value of 3x3
    neighbors = [i for i in range(9) if i != 4]
    
    # Generate the signed matrix requested
    diff_matrix = (matrix - center_val).astype(int)
    
    # 2. Build the QUBO
    # We maximize abs(diff) by minimizing -abs(diff)
    abs_diffs = {i: abs(flat[i] - center_val) for i in neighbors}
    
    # Penalty (P) ensures only one neighbor is selected
    P = max(abs_diffs.values()) * 2
    
    qubo = {}
    # Linear: (Objective + Penalty terms)
    for i in neighbors:
        qubo[(i, i)] = -abs_diffs[i] - P
        
    # Quadratic: (Penalty to prevent multiple selections)
    for i in neighbors:
        for j in neighbors:
            if i < j:
                qubo[(i, j)] = 2 * P

    # 3. Solve using ExactSolver (Local, no Leap account)
    solver = dimod.ExactSolver()
    # This checks all 256 possible neighbor combinations
    response = solver.sample_qubo(qubo)
    
    # Get the state with the lowest energy
    best_sample = response.first.sample
    
    # Extract the index where the variable was 1
    selected_indices = [k for k, v in best_sample.items() if v == 1]
    
    # Handle the edge case of a tie or no selection (though P prevents this)
    max_diff_val = abs_diffs[selected_indices[0]] if selected_indices else 0
    
    return diff_matrix, max_diff_val

# Test with a 3x3 matrix
mat = np.array([[10, 20, 30],
                [40, 50, 60],
                [70,  80, 90]])

signed_mat, largest_diff = solve_max_diff_exact(mat)

print("--- Resulting Signed Matrix ---")
print(signed_mat)
print(f"\nLargest Difference found by ExactSolver: {largest_diff}")


1.28 μs ± 694 ns per loop (mean ± std. dev. of 5 runs, 1 loop each)
Best time: 7.00121745467186e-07
--- Resulting Signed Matrix ---
[[-40 -30 -20]
 [-10   0  10]
 [ 20  30  40]]

Largest Difference found by ExactSolver: 40


In [24]:
res = %timeit -n 1 -r 5  -o [x for x in range(10)]
print(f"Best time: {res.best}")
import numpy as np
import dimod

def solve_matrix_cqm_fixed(matrix):
    # 1. Setup
    flat = matrix.flatten()
    center_val = int(flat[4])
    signed_diff_matrix = (matrix - center_val).astype(int)
    
    # 2. CQM Construction
    cqm = dimod.ConstrainedQuadraticModel()
    max_range = int(np.max(matrix) - np.min(matrix))
    max_diff_var = dimod.Integer("largest_diff", lower_bound=0, upper_bound=max_range)
    cqm.set_objective(max_diff_var)
    
    # Neighbors indices
    neighbors = [i for i in range(9) if i != 4]
    for i in neighbors:
        diff = abs(int(flat[i]) - center_val)
        cqm.add_constraint(max_diff_var >= diff, label=f"n{i}")
    
    # 3. Solve Locally
    solver = dimod.ExactCQMSolver()
    sampleset = solver.sample_cqm(cqm)
    
    # 4. FIX: Use all() to evaluate the array of boolean satisfaction
    feasible_samples = sampleset.filter(lambda row: all(row.is_satisfied))
    
    if len(feasible_samples) > 0:
        # Accessing the first (best) sample
        best_sample = feasible_samples.first.sample
        result_val = int(best_sample["largest_diff"])
    else:
        result_val = "No feasible solution"
        
    return signed_diff_matrix, result_val

# Test
mat = np.array([[10, 20, 30],
                [40, 50, 60],
                [70,  80, 90]])

signed_mat, max_diff = solve_matrix_cqm_fixed(mat)
print(f"Signed Matrix:\n{signed_mat}")
print(f"Largest Difference: {max_diff}")


1.66 μs ± 869 ns per loop (mean ± std. dev. of 5 runs, 1 loop each)
Best time: 1.00000761449337e-06
Signed Matrix:
[[-40 -30 -20]
 [-10   0  10]
 [ 20  30  40]]
Largest Difference: 40
